# Лекция: Вероятностные распределения в Python

**Дисциплина:** Введение в анализ больших данных

В этой лекции вы познакомитесь с работой с непрерывными распределениями в Python:
- нормальное распределение;
- равномерное распределение;
- распределение Стьюдента (t-распределение);
- построение гистограмм и кривых плотности.

Основные инструменты:
- **NumPy** — генерация случайных чисел и базовые операции;
- **SciPy** (`scipy.stats`) — теоретические распределения, плотность, функция распределения;
- **Matplotlib** / **Seaborn** — визуализация.

Примеры ниже **не совпадают** с формулировками лабораторного задания. Они нужны, чтобы освоить методы и атрибуты. Само задание выполните самостоятельно, опираясь на изученные приёмы.


## 0. Импорт библиотек


In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12
sns.set_style("whitegrid")

print("NumPy:", np.__version__)
print("Библиотеки загружены")


---
## Краткая теория

**Распределение** числовой случайной величины задаёт, с какими вероятностями она принимает значения на числовой оси.

Для непрерывных распределений важны:
- **плотность** $f(x)$ — `pdf` (probability density function);
- **функция распределения** $F(x) = P(X \le x)$ — `cdf`;
- **квантили** — обратная к cdf (`ppf`).

### Объекты `scipy.stats`

Типичный объект распределения (например, `stats.norm`, `stats.uniform`, `stats.t`) поддерживает методы:

| Метод | Назначение |
|-------|------------|
| `.rvs(size=n, ...)` | случайная выборка объёма n |
| `.pdf(x)` | значение плотности в точке x |
| `.cdf(x)` | функция распределения |
| `.ppf(q)` | квантиль уровня q |
| `.mean()`, `.std()`, `.var()` | теоретические моменты |

Параметры часто задаются как `loc` (сдвиг) и `scale` (масштаб). Для t-распределения ключевой параметр — `df` (степени свободы).


---
## 1. Нормальное распределение

Нормальное (гауссовское) распределение симметрично и полностью задаётся средним $\mu$ (`loc`) и стандартным отклонением $\sigma$ (`scale`).

**Пример контекста:** рост взрослых мужчин в некоторой популяции часто моделируют как $N(175,\,7)$ (см).


In [ ]:
# Фиксируем зерно генератора — результаты воспроизводимы
np.random.seed(2024)

# Выборка «ростов»: n = 30, mean = 175, sd = 7
heights = stats.norm.rvs(loc=175, scale=7, size=30)

print("Первые 10 значений:", np.round(heights, 1)[:10])
print("Выборочное среднее :", round(heights.mean(), 2))
print("Выборочное sd      :", round(heights.std(ddof=1), 2))
print("Теоретические mean/sd:", stats.norm(loc=175, scale=7).mean(),
      stats.norm(loc=175, scale=7).std())


In [ ]:
# Другая выборка: стандартное нормальное N(0, 1), n = 50
z = stats.norm.rvs(loc=0, scale=1, size=50)
# эквивалентно: np.random.normal(0, 1, 50)

print("min =", round(z.min(), 3), " max =", round(z.max(), 3))
print("mean ≈", round(z.mean(), 3), " sd ≈", round(z.std(ddof=1), 3))


### Гистограмма и теоретическая плотность

Гистограмма с `density=True` показывает оценку плотности по выборке.  
Поверх неё удобно нарисовать теоретическую кривую `.pdf`.


In [ ]:
# Выборки с разными параметрами (не те, что в лабораторной работе)
np.random.seed(7)
sample_A = stats.norm.rvs(loc=0, scale=1, size=200)    # N(0, 1)
sample_B = stats.norm.rvs(loc=5, scale=2, size=200)    # N(5, 2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, sample, mu, sigma, title in [
    (axes[0], sample_A, 0, 1, "N(0, 1), n=200"),
    (axes[1], sample_B, 5, 2, "N(5, 2), n=200"),
]:
    ax.hist(sample, bins=20, density=True, color="steelblue",
            alpha=0.7, edgecolor="white", label="гистограмма")
    xs = np.linspace(sample.min() - 1, sample.max() + 1, 300)
    ax.plot(xs, stats.norm.pdf(xs, loc=mu, scale=sigma),
            color="crimson", lw=2, label="теоретическая плотность")
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("плотность")
    ax.legend()

plt.tight_layout()
plt.show()


### Сравнение нескольких нормальных плотностей

Меняя $\mu$ и $\sigma$, получаем сдвиг и «растяжение» колокола.


In [ ]:
x = np.linspace(-8, 12, 500)

plt.figure(figsize=(10, 5))
plt.plot(x, stats.norm.pdf(x, 0, 1), label="N(0, 1)", lw=2)
plt.plot(x, stats.norm.pdf(x, 2, 1), label="N(2, 1)", lw=2)
plt.plot(x, stats.norm.pdf(x, 0, 3), label="N(0, 3)", lw=2)
plt.plot(x, stats.norm.pdf(x, -3, 1.5), label="N(-3, 1.5)", lw=2)
plt.title("Влияние параметров на форму нормальной плотности")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 2. Равномерное распределение

Случайная величина равномерно распределена на отрезке $[a,\,b]$, если плотность постоянна на этом отрезке и равна нулю вне него.

В `scipy.stats.uniform` параметры задаются так:
- `loc = a` — левая граница;
- `scale = b - a` — длина отрезка.

**Пример контекста:** ошибка округления измерения до целых сантиметров может моделироваться как $U(-0.5,\,0.5)$.


In [ ]:
np.random.seed(11)

# Равномерное на [10, 40], объём 40
a, b = 10, 40
u = stats.uniform.rvs(loc=a, scale=b - a, size=40)

print("Первые значения:", np.round(u, 2)[:8])
print("min =", round(u.min(), 2), " max =", round(u.max(), 2))
print("Выборочное среднее:", round(u.mean(), 2),
      "(теория:", (a + b) / 2, ")")


In [ ]:
# Гистограмма + теоретическая плотность U(10, 40)
fig, ax = plt.subplots()
ax.hist(u, bins=12, density=True, color="seagreen", alpha=0.7,
        edgecolor="white", label="гистограмма")
xs = np.linspace(5, 45, 300)
ax.plot(xs, stats.uniform.pdf(xs, loc=a, scale=b - a),
        color="darkred", lw=2, label="плотность U(10, 40)")
ax.set_title("Равномерное распределение на [10, 40]")
ax.set_xlabel("x")
ax.set_ylabel("плотность")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 3. Распределение Стьюдента (t)

t-распределение похоже на нормальное, но с более «тяжёлыми» хвостами.  
Единственный параметр — **число степеней свободы** `df`. При больших `df` оно почти совпадает с $N(0,1)$.

**Пример контекста:** доверительные интервалы для среднего при неизвестной дисперсии опираются на t-распределение.


In [ ]:
# Плотности t при разных df (сетка по x — произвольная, не из задания)
x = np.linspace(-6, 6, 400)

plt.figure(figsize=(10, 5))
for df, style in [(2, "-"), (4, "--"), (15, "-."), (30, ":")]:
    plt.plot(x, stats.t.pdf(x, df=df), style, lw=2, label=f"t(df={df})")

plt.plot(x, stats.norm.pdf(x, 0, 1), "k", lw=2, alpha=0.6, label="N(0, 1)")
plt.title("t-распределение при разных степенях свободы")
plt.xlabel("x")
plt.ylabel("плотность")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Генерация выборки из t(df=8)
np.random.seed(99)
t_sample = stats.t.rvs(df=8, size=80)
print("t(df=8), n=80")
print("mean ≈", round(t_sample.mean(), 3),
      " sd ≈", round(t_sample.std(ddof=1), 3))
print("Теоретические: mean =", stats.t(df=8).mean(),
      " sd ≈", round(stats.t(df=8).std(), 3))


---
## 4. Полезные приёмы

### KDE (ядерная оценка плотности) через Seaborn

Альтернатива ручному наложению `.pdf` — сглаженная оценка по данным.


In [ ]:
np.random.seed(3)
data = stats.norm.rvs(loc=50, scale=8, size=150)

plt.figure(figsize=(9, 5))
sns.histplot(data, bins=20, stat="density", color="skyblue",
             edgecolor="white", label="гистограмма")
sns.kdeplot(data, color="navy", lw=2, label="KDE (по данным)")
xs = np.linspace(data.min() - 5, data.max() + 5, 300)
plt.plot(xs, stats.norm.pdf(xs, 50, 8), "r--", lw=2, label="теория N(50, 8)")
plt.title("Гистограмма + KDE + теоретическая плотность")
plt.xlabel("x")
plt.legend()
plt.tight_layout()
plt.show()


### Краткая сводка по выборке


In [ ]:
def describe_sample(sample, name="Выборка"):
    sample = np.asarray(sample, dtype=float)
    print(f"=== {name} ===")
    print(f"n              : {len(sample)}")
    print(f"минимум        : {sample.min():.4f}")
    print(f"максимум       : {sample.max():.4f}")
    print(f"среднее        : {sample.mean():.4f}")
    print(f"медиана        : {np.median(sample):.4f}")
    print(f"ст. отклонение : {sample.std(ddof=1):.4f}")
    print(f"дисперсия      : {sample.var(ddof=1):.4f}")
    print()

describe_sample(heights, "Рост N(175, 7), n=30")
describe_sample(u, "Uniform[10, 40], n=40")
describe_sample(t_sample, "t(df=8), n=80")


---
## Шпаргалка по методам (Python)

| Задача | Код |
|--------|-----|
| Выборка из $N(\mu,\sigma)$ | `stats.norm.rvs(loc=mu, scale=sigma, size=n)` |
| Плотность $N(\mu,\sigma)$ | `stats.norm.pdf(x, loc=mu, scale=sigma)` |
| CDF / квантиль | `stats.norm.cdf(x, ...)`, `stats.norm.ppf(q, ...)` |
| Выборка из $U(a,b)$ | `stats.uniform.rvs(loc=a, scale=b-a, size=n)` |
| Плотность $U(a,b)$ | `stats.uniform.pdf(x, loc=a, scale=b-a)` |
| Выборка из $t(df)$ | `stats.t.rvs(df=df, size=n)` |
| Плотность $t(df)$ | `stats.t.pdf(x, df=df)` |
| Гистограмма плотности | `plt.hist(x, bins=..., density=True)` |
| Воспроизводимость | `np.random.seed(число)` |

---
## Что сделать после лекции

1. Повторите генерацию выборок и построение графиков с **другими** параметрами.
2. Откройте лабораторное задание и выполните его **самостоятельно**, используя методы из таблицы выше.
3. Документация: [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html).

Удачи в работе с распределениями!
